# Gold Layer - Bussiness Metrics & Analytics

## Load Silver tables

In [0]:

taxi_trip_silver = spark.table(
    "taxicatalog.`taxi_project-schema`.taxi_trip_silver"
)

taxi_location_silver = spark.table(
    "taxicatalog.`taxi_project-schema`.taxi_location_silver"
)

print("Silver tables loaded successfully.")
print(f"Trip records: {taxi_trip_silver.count()}")
print(f"Location records: {taxi_location_silver.count()}")

## Daily Taxi Metrics

This table will give us a business-level view of taxi activity by day.
- Total trips
- Total revenue
- Average fare
- Average tip
- Total distance
- Average trip distance
- Average trip duration
- Total passengers

In [0]:

from pyspark.sql.functions import (
    count,
    sum,
    avg,
    round
)

daily_taxi_metrics = (
    taxi_trip_silver
    .groupBy("pickup_date")
    .agg(
        count("trip_id").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        sum("fare_amount").alias("total_fare"),

        sum("tip_amount").alias("total_tips"),

        sum("trip_distance_miles").alias("total_distance_miles"),

        sum("passenger_count").alias("total_passengers"),

        round(avg("fare_amount"), 2).alias("average_fare"),

        round(avg("tip_amount"), 2).alias("average_tip"),

        round(avg("trip_distance_miles"), 2).alias("average_trip_distance_miles"),

        round(avg("trip_duration_minutes"), 2).alias("average_trip_duration_minutes")
    )
    .orderBy("pickup_date")
)

display(daily_taxi_metrics)

## Quick validation

In [0]:

print(f"Gold daily records: {daily_taxi_metrics.count()}")

display(
    daily_taxi_metrics
    .select(
        "pickup_date",
        "total_trips",
        "total_revenue",
        "average_fare",
        "average_trip_distance_miles",
        "average_trip_duration_minutes"
    )
)

## Save Daily Metrics

In [0]:
(
    daily_taxi_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "taxicatalog.`taxi_project-schema`.daily_taxi_metrics"
    )
)

## Verify

In [0]:
display(
    spark.table(
        "taxicatalog.`taxi_project-schema`.daily_taxi_metrics"
    )
)

## Location Performance

Analyze each pickup location and calculate:
- Number of trips
- Revenue
- Average fare
- Average trip distance
- Average trip duration
- Total passengers
- Average tip

In [0]:

from pyspark.sql.functions import count, sum, avg, round

location_performance = (
    taxi_trip_silver
    .groupBy(
        "pickup_location_id",
        "pickup_borough",
        "pickup_zone"
    )
    .agg(
        count("trip_id").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        sum("passenger_count").alias("total_passengers"),

        round(avg("fare_amount"), 2).alias("average_fare"),

        round(avg("tip_amount"), 2).alias("average_tip"),

        round(avg("trip_distance_miles"), 2)
            .alias("average_trip_distance_miles"),

        round(avg("trip_duration_minutes"), 2)
            .alias("average_trip_duration_minutes")
    )
    .orderBy("total_revenue", ascending=False)
)

display(location_performance)

## Save Location Performance

In [0]:
(
    location_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "taxicatalog.`taxi_project-schema`.location_performance"
    )
)

## Verify

In [0]:
display(
    spark.table(
        "taxicatalog.`taxi_project-schema`.location_performance"
    )
)

## Hourly Metrics

In [0]:

hourly_taxi_metrics = (
    taxi_trip_silver
    .groupBy(
        "pickup_hour"
    )
    .agg(
        count("trip_id").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        sum("passenger_count").alias("total_passengers"),

        round(avg("fare_amount"), 2)
            .alias("average_fare"),

        round(avg("trip_distance_miles"), 2)
            .alias("average_trip_distance_miles"),

        round(avg("trip_duration_minutes"), 2)
            .alias("average_trip_duration_minutes")
    )
    .orderBy("pickup_hour")
)

display(hourly_taxi_metrics)

## Save hourtly metrics to table

In [0]:
(
    hourly_taxi_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "taxicatalog.`taxi_project-schema`.hourly_taxi_metrics"
    )
)

## Verify

In [0]:
display(
    spark.table(
        "taxicatalog.`taxi_project-schema`.hourly_taxi_metrics"
    )
)

## Payment Type Analysis

In [0]:
from pyspark.sql.functions import count, sum, avg, round

payment_analysis = (
    taxi_trip_silver
    .groupBy("payment_type")
    .agg(
        count("trip_id").alias("total_trips"),
        sum("total_amount").alias("total_revenue"),
        sum("fare_amount").alias("total_fare"),
        sum("tip_amount").alias("total_tips"),
        round(avg("fare_amount"), 2).alias("average_fare"),
        round(avg("tip_amount"), 2).alias("average_tip"),
        round(avg("total_amount"), 2).alias("average_total_amount")
    )
    .orderBy("total_revenue", ascending=False)
)

display(payment_analysis)

## Save Payment Analysis

In [0]:
(
    payment_analysis.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "taxicatalog.`taxi_project-schema`.payment_analysis"
    )
)

## Verify

In [0]:
display(
    spark.table(
        "taxicatalog.`taxi_project-schema`.payment_analysis"
    )
)

## Final Verfication 

In [0]:
spark.sql("""
SHOW TABLES IN taxicatalog.`taxi_project-schema`
""").display()